# image segmentation in python using pixel classification
a notebook to learn image segmentation using labeled images

in Colab, we need to connect to a GPU

on the to right, go to change runtime type and select T4 GPU

In [ ]:
pip install scikit-image

In [ ]:
pip install matplotlib

In [ ]:
!pip install apoc --no-deps
!pip install "scikit-learn" "pyclesperanto-prototype" "pandas" "numpy==2.4.4"

In [ ]:
from matplotlib import pyplot as plt
import apoc
from skimage.io import imread, imsave
import numpy as np

In [ ]:
import os
# Clone the repo if not already in Colab
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/NBCimageAnalysis'):
        !git clone https://github.com/FilLieb/NBCimageAnalysis.git
    os.chdir('/content/NBCimageAnalysis/learning/')
    print(os.listdir('.'))

define the paths were images and labeled masks are saved

In [ ]:
image_folder = '../training/images/'
masks_folder = '../training/masks/'

print("Image folder path:", image_folder)
print("Masks folder path:", masks_folder)

display one image and its corresponding mask from the folder to verify paths are correct


In [ ]:
image_path = os.path.join(image_folder, os.listdir(image_folder)[0])
image = imread(image_path)

masks_path = os.path.join(masks_folder, os.listdir(masks_folder)[0])
masks = imread(masks_path)


f, a = plt.subplots(1,3, figsize=(15,5))
a[0].imshow(image, cmap='gray')
a[1].imshow(masks, vmin=0, vmax=2)
a[2].imshow(image, cmap='gray')
a[2].contour(masks, colors='r', linewidths=0.5)

plt.show()

define were the model is saved

In [ ]:
cl_filename = '../training/models/object_model.cl'

setup classifier

In [ ]:
segmenter = apoc.ObjectSegmenter(opencl_filename=cl_filename,
                                     max_depth=5,
                                     num_ensembles=1000)

setup feature sets used for training

In [ ]:
features = apoc.PredefinedFeatureSet.small_dog_log.value + " " + \
           apoc.PredefinedFeatureSet.medium_dog_log.value + " " + \
           apoc.PredefinedFeatureSet.large_dog_log.value

train classifier on folders

In [ ]:
apoc.erase_classifier(cl_filename) # delete it if it was existing before
apoc.train_classifier_from_image_folders(
        segmenter,
        features,
        image = image_folder,
        ground_truth = masks_folder)

print("Training completed and model saved to " + cl_filename)

finally we can test how the model performed

In [ ]:
segmenter = apoc.ObjectSegmenter(opencl_filename=cl_filename)
image = imread(image_path)

labels = segmenter.predict(image)

f, a = plt.subplots(1,3, figsize=(15,5))
a[0].imshow(image, cmap='gray')
a[1].imshow(labels, vmin=0, vmax=2)
a[2].imshow(image, cmap='gray')
a[2].contour(labels, colors='r', linewidths=0.5)

plt.show()